# ARC spatial-program offline inference

This notebook loads the fixed Qwen3 LoRA adapter trained by `arc-spatial-program-sft`, infers two executable spatial programs for every ARC-AGI-2 test input, checks each program against all demonstrations, and writes `/kaggle/working/submission.json`.

It runs fully offline. Attach the ARC Prize 2026 ARC-AGI-2 competition, the Qwen base model, the Python wheelhouse, and the completed training notebook output. Use Kaggle's L4 accelerator.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

WHEELHOUSE = Path("/kaggle/input/datasets/aishikai/offline-unsloth-trl-wheelhouse-py312-cu128")
assert (WHEELHOUSE / "requirements.in").exists(), "Attach aishikai/offline-unsloth-trl-wheelhouse-py312-cu128"
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-index",
    "--find-links", str(WHEELHOUSE), "-r", str(WHEELHOUSE / "requirements.in"),
], check=True)
print("Offline dependencies installed.")


In [ ]:
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch

SEED = 3407
MAX_SEQ_LENGTH = 12_288
MAX_NEW_TOKENS = 160


def find_parent(filename, required_sibling=None, preferred=()):
    matches = []
    for path in Path("/kaggle/input").glob(f"**/{filename}"):
        if required_sibling and not (path.parent / required_sibling).exists():
            continue
        score = sum(part in str(path) for part in preferred)
        matches.append((-score, len(path.parts), path.parent))
    return str(sorted(matches)[0][2]) if matches else None


BASE_MODEL = find_parent(
    "config.json", preferred=("qwen3-4b-instruct-2507-unsloth-4bit", "bnb-4bit")
)
ADAPTER = find_parent(
    "adapter_model.safetensors", required_sibling="adapter_config.json", preferred=("arc-spatial-program-sft", "outputs/adapters")
)
TEST_FILE = next(Path("/kaggle/input").glob("**/arc-agi_test_challenges.json"), None)

assert BASE_MODEL, "Attach aishikai/qwen3-4b-instruct-2507-unsloth-4bit"
assert ADAPTER, "Attach the output of aishikai/arc-spatial-program-sft"
assert TEST_FILE, "Attach the ARC Prize 2026 ARC-AGI-2 competition"
assert Path(ADAPTER, "adapter_model.safetensors").stat().st_size > 100_000_000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print({"base_model": BASE_MODEL, "adapter": ADAPTER, "test_file": str(TEST_FILE)})


In [ ]:
DIRECTIONS = {
    "up": (-1, 0),
    "down": (1, 0),
    "left": (0, -1),
    "right": (0, 1),
}


def as_grid(grid):
    a = np.asarray(grid, dtype=int)
    if a.ndim != 2 or not (1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30):
        raise ValueError("grid must be rectangular and at most 30x30")
    if a.min() < 0 or a.max() > 9:
        raise ValueError("colors must be 0..9")
    return a


def background(grid):
    a = as_grid(grid)
    counts = Counter(a.ravel().tolist())
    return min(counts, key=lambda color: (-counts[color], color))


def components(grid):
    a = as_grid(grid)
    bg = background(a)
    seen = set()
    out = []
    for r in range(a.shape[0]):
        for c in range(a.shape[1]):
            color = int(a[r, c])
            if color == bg or (r, c) in seen:
                continue
            q = [(r, c)]
            seen.add((r, c))
            cells = []
            while q:
                rr, cc = q.pop()
                cells.append((rr, cc))
                for dr, dc in DIRECTIONS.values():
                    nr, nc = rr + dr, cc + dc
                    if (
                        0 <= nr < a.shape[0]
                        and 0 <= nc < a.shape[1]
                        and (nr, nc) not in seen
                        and int(a[nr, nc]) == color
                    ):
                        seen.add((nr, nc))
                        q.append((nr, nc))
            rs = [x[0] for x in cells]
            cs = [x[1] for x in cells]
            out.append({
                "cells": cells,
                "color": color,
                "size": len(cells),
                "r": min(rs),
                "c": min(cs),
                "h": max(rs) - min(rs) + 1,
                "w": max(cs) - min(cs) + 1,
            })
    return out


def select_object(grid, selector):
    objs = components(grid)
    if not objs:
        raise ValueError("no foreground objects")
    keys = {
        "largest": lambda o: (-o["size"], o["r"], o["c"]),
        "smallest": lambda o: (o["size"], o["r"], o["c"]),
        "topmost": lambda o: (o["r"], o["c"], -o["size"]),
        "bottommost": lambda o: (-(o["r"] + o["h"]), o["c"], -o["size"]),
        "leftmost": lambda o: (o["c"], o["r"], -o["size"]),
        "rightmost": lambda o: (-(o["c"] + o["w"]), o["r"], -o["size"]),
    }
    if selector not in keys:
        raise ValueError(f"unknown selector: {selector}")
    return sorted(objs, key=keys[selector])[0]


def crop_foreground(grid):
    a = as_grid(grid)
    bg = background(a)
    cells = np.argwhere(a != bg)
    if not len(cells):
        return a.tolist()
    r0, c0 = cells.min(axis=0)
    r1, c1 = cells.max(axis=0)
    return a[r0:r1 + 1, c0:c1 + 1].tolist()


def recolor_object(grid, selector, color):
    a = as_grid(grid).copy()
    obj = select_object(a, selector)
    for r, c in obj["cells"]:
        a[r, c] = int(color)
    return a.tolist()


def recolor_foreground(grid, color):
    a = as_grid(grid).copy()
    bg = background(a)
    a[a != bg] = int(color)
    return a.tolist()


def move_object(grid, selector, direction, steps):
    a = as_grid(grid).copy()
    bg = background(a)
    obj = select_object(a, selector)
    dr, dc = DIRECTIONS[direction]
    moved = [(r + dr * steps, c + dc * steps) for r, c in obj["cells"]]
    own = set(obj["cells"])
    for r, c in moved:
        if not (0 <= r < a.shape[0] and 0 <= c < a.shape[1]):
            raise ValueError("move leaves grid")
        if int(a[r, c]) != bg and (r, c) not in own:
            raise ValueError("move collides")
    for r, c in obj["cells"]:
        a[r, c] = bg
    for r, c in moved:
        a[r, c] = obj["color"]
    return a.tolist()


def extract_object(grid, selector):
    a = as_grid(grid)
    bg = background(a)
    obj = select_object(a, selector)
    out = np.full((obj["h"], obj["w"]), bg, dtype=int)
    for r, c in obj["cells"]:
        out[r - obj["r"], c - obj["c"]] = obj["color"]
    return out.tolist()


def complete_symmetry(grid, axis):
    a = as_grid(grid).copy()
    bg = background(a)
    source = a.copy()
    for r in range(a.shape[0]):
        for c in range(a.shape[1]):
            if int(source[r, c]) == bg:
                continue
            rr, cc = (r, a.shape[1] - 1 - c) if axis == "vertical" else (a.shape[0] - 1 - r, c)
            if int(a[rr, cc]) == bg:
                a[rr, cc] = source[r, c]
    return a.tolist()


def connect_markers(grid):
    a = as_grid(grid).copy()
    singletons = [o for o in components(a) if o["size"] == 1]
    pairs = []
    for i, first in enumerate(singletons):
        for second in singletons[i + 1:]:
            if first["color"] != second["color"]:
                continue
            p = first["cells"][0]
            q = second["cells"][0]
            if p[0] == q[0] or p[1] == q[1]:
                pairs.append((p, q, first["color"]))
    if len(pairs) != 1:
        raise ValueError("expected one aligned equal-color marker pair")
    (r1, c1), (r2, c2), color = pairs[0]
    if r1 == r2:
        a[r1, min(c1, c2):max(c1, c2) + 1] = color
    else:
        a[min(r1, r2):max(r1, r2) + 1, c1] = color
    return a.tolist()


def trace_path(grid, orientation):
    a = as_grid(grid)
    bg = background(a)
    cells = {tuple(x) for x in np.argwhere(a != bg)}
    if not cells:
        raise ValueError("empty path")
    neighbors = {
        p: [
            (p[0] + dr, p[1] + dc)
            for dr, dc in DIRECTIONS.values()
            if (p[0] + dr, p[1] + dc) in cells
        ]
        for p in cells
    }
    endpoints = sorted(p for p, ns in neighbors.items() if len(ns) == 1)
    if len(endpoints) != 2 or any(len(ns) > 2 for ns in neighbors.values()):
        raise ValueError("foreground is not one simple path")
    order = []
    previous = None
    current = endpoints[0]
    while current is not None:
        order.append(current)
        nxt = [p for p in neighbors[current] if p != previous]
        previous, current = current, (nxt[0] if nxt else None)
    if len(order) != len(cells):
        raise ValueError("path is disconnected")
    values = [int(a[r, c]) for r, c in order]
    return ([values] if orientation == "row" else [[x] for x in values])


def copy_marker_color(grid, selector):
    a = as_grid(grid)
    target = select_object(a, selector)
    markers = [o for o in components(a) if o["size"] == 1 and o["cells"] != target["cells"]]
    if len(markers) != 1:
        raise ValueError("expected one singleton marker")
    return recolor_object(a, selector, markers[0]["color"])


def upscale(grid, factor):
    a = as_grid(grid)
    return np.repeat(np.repeat(a, factor, axis=0), factor, axis=1).tolist()


def apply_program(grid, program):
    out = as_grid(grid).tolist()
    for step in program:
        op = step["op"]
        if op == "rotate":
            out = np.rot90(as_grid(out), -int(step["k"])).tolist()
        elif op == "flip":
            out = (np.fliplr(as_grid(out)) if step["axis"] == "vertical" else np.flipud(as_grid(out))).tolist()
        elif op == "recolor_object":
            out = recolor_object(out, step["selector"], step["color"])
        elif op == "recolor_foreground":
            out = recolor_foreground(out, step["color"])
        elif op == "move_object":
            out = move_object(out, step["selector"], step["direction"], step["steps"])
        elif op == "extract_object":
            out = extract_object(out, step["selector"])
        elif op == "crop_foreground":
            out = crop_foreground(out)
        elif op == "complete_symmetry":
            out = complete_symmetry(out, step["axis"])
        elif op == "connect_markers":
            out = connect_markers(out)
        elif op == "trace_path":
            out = trace_path(out, step["orientation"])
        elif op == "copy_marker_color":
            out = copy_marker_color(out, step["selector"])
        elif op == "upscale":
            out = upscale(out, int(step["factor"]))
        else:
            raise ValueError(f"unknown operation: {op}")
        as_grid(out)
    return out


def canonical_program(program):
    return json.dumps(program, separators=(",", ":"))


In [ ]:
DSL_SPEC = """Programs are JSON arrays executed left to right.
Allowed operations:
{"op":"rotate","k":1|2|3}
{"op":"flip","axis":"horizontal"|"vertical"}
{"op":"recolor_object","selector":SELECTOR,"color":0..9}
{"op":"recolor_foreground","color":0..9}
{"op":"move_object","selector":SELECTOR,"direction":"up"|"down"|"left"|"right","steps":1|2}
{"op":"extract_object","selector":SELECTOR}
{"op":"crop_foreground"}
{"op":"complete_symmetry","axis":"horizontal"|"vertical"}
{"op":"connect_markers"}
{"op":"trace_path","orientation":"row"|"column"}
{"op":"copy_marker_color","selector":SELECTOR}
{"op":"upscale","factor":2|3}
SELECTOR is largest, smallest, topmost, bottommost, leftmost, or rightmost.
Colors are integers 0..9. Return only the JSON array."""

SYSTEM_PROMPT = (
    "Infer the shortest valid spatial program that explains every demonstration. "
    "Return only one JSON array. Do not return prose or the query grid."
)


def render_grid(grid):
    return "\n".join("".join(str(int(x)) for x in row) for row in grid)


def build_user_prompt(demos, query):
    parts = [DSL_SPEC, "--- DEMONSTRATIONS ---"]
    for index, example in enumerate(demos, 1):
        inp, out = example["input"], example["output"]
        parts.append(f"Demo {index} input ({len(inp)}x{len(inp[0])}):\n{render_grid(inp)}")
        parts.append(f"Demo {index} output ({len(out)}x{len(out[0])}):\n{render_grid(out)}")
    parts.append(f"--- QUERY ---\nInput ({len(query)}x{len(query[0])}):\n{render_grid(query)}")
    parts.append("Return the program only.")
    return "\n\n".join(parts)


def extract_program(text):
    start = text.find("[")
    if start < 0:
        raise ValueError("no JSON array")
    program, _ = json.JSONDecoder().raw_decode(text[start:])
    if not isinstance(program, list):
        raise ValueError("program is not a list")
    return program


def solves_demos(program, demos):
    try:
        return all(apply_program(row["input"], program) == row["output"] for row in demos)
    except (KeyError, TypeError, ValueError):
        return False


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    local_files_only=True,
)
FastLanguageModel.for_inference(model)
tokenizer.truncation_side = "left"
print(f"Loaded adapter on {model.device}")


In [ ]:
@torch.inference_mode()
def generate_program(demos, query, sample, seed):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(demos, query)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH - MAX_NEW_TOKENS,
    ).to(model.device)
    torch.manual_seed(seed)
    kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": sample,
        "use_cache": True,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if sample:
        kwargs.update(temperature=0.6, top_p=0.9)
    output = model.generate(**inputs, **kwargs)
    text = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return extract_program(text)


def solve_test(task_id, demos, query, test_index):
    fallback = as_grid(query).tolist()
    predictions = []
    for sample in (False, True):
        try:
            seed = SEED + int(task_id, 16) + test_index * 2 + int(sample)
            program = generate_program(demos, query, sample, seed)
            if solves_demos(program, demos):
                grid = apply_program(query, program)
                if grid not in predictions:
                    predictions.append(grid)
        except (json.JSONDecodeError, KeyError, TypeError, ValueError):
            pass
    while len(predictions) < 2:
        predictions.append(predictions[0] if predictions else fallback)
    return {"attempt_1": predictions[0], "attempt_2": predictions[1]}


challenges = json.loads(TEST_FILE.read_text())
submission = {}
total = sum(len(task["test"]) for task in challenges.values())
done = 0
for task_id, task in challenges.items():
    submission[task_id] = []
    for test_index, test in enumerate(task["test"]):
        submission[task_id].append(solve_test(task_id, task["train"], test["input"], test_index))
        done += 1
        if done % 10 == 0:
            print(f"{done}/{total}")


In [ ]:
assert set(submission) == set(challenges)
for task_id, task in challenges.items():
    assert len(submission[task_id]) == len(task["test"])
    for prediction in submission[task_id]:
        assert set(prediction) == {"attempt_1", "attempt_2"}
        as_grid(prediction["attempt_1"])
        as_grid(prediction["attempt_2"])

SUBMISSION_FILE = Path("/kaggle/working/submission.json")
SUBMISSION_FILE.write_text(json.dumps(submission, separators=(",", ":")))
print({"tasks": len(submission), "test_outputs": total, "file": str(SUBMISSION_FILE), "bytes": SUBMISSION_FILE.stat().st_size})
